# CSV 데이터 개요, EDA, 시각화 노트북

이 노트북은 CSV 파일을 불러와 데이터 개요를 출력하고, 기본적인 EDA와 시각화를 자동으로 수행합니다.

실행 순서:
1. 아래 셀부터 순서대로 실행합니다.
2. CSV 선택창이 열리면 분석할 파일을 선택합니다.
3. 각 시각화와 통계 요약을 확인합니다.


In [ ]:
import importlib
import subprocess
import sys

required_packages = {
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

print("Environment is ready.")


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib_cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False

print("Libraries imported.")


In [ ]:
def choose_csv_file():
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        file_path = filedialog.askopenfilename(
            title="CSV 파일 선택",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
        )
        root.destroy()
        if file_path:
            return Path(file_path)
    except Exception as exc:
        print(f"파일 선택창을 열지 못했습니다: {exc}")

    manual_path = input("CSV 파일 경로를 직접 입력하세요: ").strip().strip('"')
    if not manual_path:
        raise ValueError("CSV 파일 경로가 입력되지 않았습니다.")
    return Path(manual_path)


def load_csv_with_fallback(file_path):
    encodings = ["utf-8", "utf-8-sig", "cp949", "euc-kr", "latin1"]
    last_error = None
    for encoding in encodings:
        try:
            data = pd.read_csv(file_path, encoding=encoding)
            return data, encoding
        except Exception as exc:
            last_error = exc
    raise last_error


csv_path = choose_csv_file()
if not csv_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {csv_path}")

df, used_encoding = load_csv_with_fallback(csv_path)
print(f"Loaded file: {csv_path}")
print(f"Encoding used: {used_encoding}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()


## 1. 데이터 개요


In [ ]:
overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_ratio": (df.isna().mean().values * 100).round(2),
    "unique_count": df.nunique(dropna=True).values,
})

duplicate_count = int(df.duplicated().sum())

print(f"행 수: {len(df):,}")
print(f"열 수: {df.shape[1]:,}")
print(f"중복 행 수: {duplicate_count:,}")

overview.sort_values(["null_ratio", "unique_count"], ascending=[False, False]).reset_index(drop=True)


In [ ]:
display(df.head())
display(df.tail())

numeric_summary = df.describe(include=[np.number]).T
categorical_summary = df.describe(include=["object", "category", "bool"]).T

if not numeric_summary.empty:
    print("수치형 변수 요약")
    display(numeric_summary)
else:
    print("수치형 변수가 없습니다.")

if not categorical_summary.empty:
    print("범주형 변수 요약")
    display(categorical_summary)
else:
    print("범주형 변수가 없습니다.")


## 2. 전처리용 컬럼 분류


In [ ]:
df_eda = df.copy()
potential_datetime_cols = []

for col in df_eda.select_dtypes(include=["object", "string"]).columns:
    converted = pd.to_datetime(df_eda[col], errors="coerce", format="mixed")
    valid_ratio = converted.notna().mean()
    if valid_ratio >= 0.8:
        df_eda[col] = converted
        potential_datetime_cols.append(col)

numeric_cols = df_eda.select_dtypes(include=[np.number]).columns.tolist()
datetime_cols = df_eda.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
categorical_cols = [col for col in df_eda.columns if col not in numeric_cols + datetime_cols]

print("수치형 컬럼:", numeric_cols if numeric_cols else "없음")
print("날짜형 컬럼:", datetime_cols if datetime_cols else "없음")
print("범주형 컬럼:", categorical_cols if categorical_cols else "없음")


## 3. 결측치 분석


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("결측치가 없습니다.")
else:
    missing_ratio = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"missing_count": missing, "missing_ratio(%)": missing_ratio})
    display(missing_df)

    plt.figure(figsize=(12, max(4, len(missing_df) * 0.4)))
    sns.barplot(x=missing_df["missing_ratio(%)"], y=missing_df.index, color="#4C72B0")
    plt.title("컬럼별 결측치 비율")
    plt.xlabel("Missing Ratio (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()


## 4. 수치형 데이터 EDA 및 시각화


In [ ]:
if not numeric_cols:
    print("수치형 컬럼이 없어 해당 시각화를 건너뜁니다.")
else:
    display(df_eda[numeric_cols].corr(numeric_only=True))

    limited_numeric_cols = numeric_cols[:6]
    fig, axes = plt.subplots(len(limited_numeric_cols), 2, figsize=(14, 4 * len(limited_numeric_cols)))

    if len(limited_numeric_cols) == 1:
        axes = np.array([axes])

    for idx, col in enumerate(limited_numeric_cols):
        sns.histplot(df_eda[col].dropna(), kde=True, ax=axes[idx, 0], color="#4C72B0")
        axes[idx, 0].set_title(f"{col} 분포")

        sns.boxplot(x=df_eda[col], ax=axes[idx, 1], color="#55A868")
        axes[idx, 1].set_title(f"{col} 박스플롯")

    plt.tight_layout()
    plt.show()

    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 8))
        corr = df_eda[numeric_cols].corr(numeric_only=True)
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", square=True)
        plt.title("수치형 변수 상관관계 히트맵")
        plt.tight_layout()
        plt.show()


## 5. 범주형 데이터 EDA 및 시각화


In [ ]:
if not categorical_cols:
    print("범주형 컬럼이 없어 해당 시각화를 건너뜁니다.")
else:
    for col in categorical_cols[:4]:
        print(f"상위 빈도: {col}")
        display(df_eda[col].value_counts(dropna=False).head(10).to_frame("count"))

        plt.figure(figsize=(10, 5))
        top_values = df_eda[col].astype(str).fillna("Missing").value_counts().head(10)
        sns.barplot(x=top_values.values, y=top_values.index, color="#C44E52")
        plt.title(f"{col} 상위 10개 범주")
        plt.xlabel("Count")
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()


## 6. 날짜형 데이터 시각화


In [ ]:
if not datetime_cols:
    print("날짜형 컬럼이 없어 해당 시각화를 건너뜁니다.")
else:
    for col in datetime_cols[:3]:
        temp = df_eda[col].dropna()
        if temp.empty:
            continue

        plt.figure(figsize=(12, 4))
        temp.dt.to_period("M").astype(str).value_counts().sort_index().plot(kind="line", marker="o")
        plt.title(f"{col} 월별 건수 추이")
        plt.xlabel("Month")
        plt.ylabel("Count")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 7. 인사이트 체크 포인트

아래 항목을 보며 데이터를 해석해보세요.
- 결측치가 많은 컬럼은 무엇인가?
- 이상치가 보이는 수치형 컬럼은 무엇인가?
- 특정 범주에 데이터가 쏠려 있는가?
- 상관관계가 높은 수치형 변수 쌍이 있는가?
- 날짜 흐름에 따라 증가/감소 패턴이 있는가?
